# Times Series Building from bakery_sales_top12

## Prepare dataframe

We start from the transaction-level file bakery_sales_top12.csv where each row is one purchase (one product, its quantity sold, one time).

**Goal**: For each day and each product (article), compute
	•	daily quantity sold = sum of Quantity
	•	daily revenue = sum of Quantity * unit_price


**Output** as 12 time series (one per product), in a “wide” table:
	•	rows = days
	•	columns = products

In [2]:
import pandas as pd

path = "data_tmp/bakery_sales_top12.csv"
df = pd.read_csv(path)

print(df.shape)
df.head()

(143702, 7)


,Unnamed: 0,date,time,ticket_number,article,Quantity,unit_price
0,0,2021-01-02,08:38,150040.0,BAGUETTE,1.0,"0,90 €"
1,1,2021-01-02,08:38,150040.0,PAIN AU CHOCOLAT,3.0,"1,20 €"
2,4,2021-01-02,09:14,150041.0,PAIN AU CHOCOLAT,2.0,"1,20 €"
3,8,2021-01-02,09:25,150042.0,TRADITIONAL BAGUETTE,5.0,"1,20 €"
4,11,2021-01-02,09:25,150043.0,BAGUETTE,2.0,"0,90 €"


In [3]:
# cleaning: drop useless index column, parse date, clean numeric fields

# 1) Drop the useless index column (it can be named 'Unnamed : 0' or similar)
df = df.drop(columns=["Unnamed: 0"])    

# 2) Parse date as datetime (we keep it as a date index later)
df["date"] = pd.to_datetime(df["date"], errors="coerce")



# 3) unit_price: convert strings like "1,05 €" to float 1.05
#    - remove euro symbol, spaces, non-breaking spaces
#    - replace comma with dot
df["unit_price"] = (
    df["unit_price"]
    .astype(str)
    .str.replace("€", "", regex=False)
    .str.replace("\xa0", "", regex=False)  # non-breaking spaces
    .str.replace(" ", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")


df.head()

,date,time,ticket_number,article,Quantity,unit_price
0,2021-01-02,08:38,150040.0,BAGUETTE,1.0,0.9
1,2021-01-02,08:38,150040.0,PAIN AU CHOCOLAT,3.0,1.2
2,2021-01-02,09:14,150041.0,PAIN AU CHOCOLAT,2.0,1.2
3,2021-01-02,09:25,150042.0,TRADITIONAL BAGUETTE,5.0,1.2
4,2021-01-02,09:25,150043.0,BAGUETTE,2.0,0.9


In [4]:
# Create revenue per row
df["revenue"] = df["Quantity"] * df["unit_price"]

df[["date", "time", "ticket_number", "article", "Quantity", "unit_price", "revenue"]].head()

,date,time,ticket_number,article,Quantity,unit_price,revenue
0,2021-01-02,08:38,150040.0,BAGUETTE,1.0,0.9,0.9
1,2021-01-02,08:38,150040.0,PAIN AU CHOCOLAT,3.0,1.2,3.6
2,2021-01-02,09:14,150041.0,PAIN AU CHOCOLAT,2.0,1.2,2.4
3,2021-01-02,09:25,150042.0,TRADITIONAL BAGUETTE,5.0,1.2,6.0
4,2021-01-02,09:25,150043.0,BAGUETTE,2.0,0.9,1.8


## Aggregate per day and per product

Now we group by:
	•	date (day)
	•	article (product)

We compute:
	•	sum of quantities
	•	sum of revenue (turnover / CA)


In [5]:
# PYTHON — Daily aggregation (long format)
daily = (
    df.groupby(["date", "article"], as_index=False)
      .agg(
          daily_quantity=("Quantity", "sum"),
          daily_revenue=("revenue", "sum")
      )
)

daily.head(10)

,date,article,daily_quantity,daily_revenue
0,2021-01-02,BAGUETTE,46.0,41.40
1,2021-01-02,BANETTE,40.0,42.00
2,2021-01-02,BOULE 400G,11.0,16.50
3,2021-01-02,CAMPAGNE,10.0,18.00
4,2021-01-02,CEREAL BAGUETTE,19.0,23.75
5,2021-01-02,COOKIE,8.0,8.00
6,2021-01-02,CROISSANT,66.0,72.60
7,2021-01-02,PAIN AU CHOCOLAT,48.0,57.60
8,2021-01-02,SPECIAL BREAD,8.0,19.20
9,2021-01-02,TRADITIONAL BAGUETTE,128.0,153.60


## Build the 12 time series in “wide” format

We pivot so that:
	•	index = date
	•	columns = product names
	•	values = daily_quantity (and another table for daily_revenue)

Missing days/products will be filled with 0.

In [ ]:
# Pivot to wide format (quantities)
qty_ts = daily.pivot(index="date", columns="article", values="daily_quantity").fillna(0)

# Pivot to wide format (revenue)
rev_ts = daily.pivot(index="date", columns="article", values="daily_revenue").fillna(0)

# Sort by date
qty_ts = qty_ts.sort_index()
rev_ts = rev_ts.sort_index()

print("Quantity time series shape:", qty_ts.shape)
print("Revenue time series shape: ", rev_ts.shape)


Quantity time series shape: (600, 12)
Revenue time series shape:  (600, 12)


article,BAGUETTE,BANETTE,BOULE 400G,CAMPAGNE,CEREAL BAGUETTE,COOKIE,CROISSANT,ECLAIR,PAIN AU CHOCOLAT,SPECIAL BREAD,TARTELETTE,TRADITIONAL BAGUETTE
date,,,,,,,,,,,,
2021-01-02,46.0,40.0,11.0,10.0,19.0,8.0,66.0,0.0,48.0,8.0,0.0,128.0
2021-01-03,35.0,35.0,11.0,9.0,16.0,2.0,59.0,0.0,45.0,7.0,0.0,171.0
2021-01-04,30.0,24.0,5.0,6.0,7.0,2.0,17.0,0.0,17.0,7.0,0.0,128.0
2021-01-05,29.0,26.0,6.0,3.0,7.0,6.0,12.0,0.0,6.0,7.0,0.0,99.0
2021-01-07,28.0,21.0,3.0,3.0,7.0,3.0,15.0,0.0,15.0,6.0,0.0,109.0


In [8]:
display(qty_ts.head())
display(rev_ts.head())

article,BAGUETTE,BANETTE,BOULE 400G,CAMPAGNE,CEREAL BAGUETTE,COOKIE,CROISSANT,ECLAIR,PAIN AU CHOCOLAT,SPECIAL BREAD,TARTELETTE,TRADITIONAL BAGUETTE
date,,,,,,,,,,,,
2021-01-02,46.0,40.0,11.0,10.0,19.0,8.0,66.0,0.0,48.0,8.0,0.0,128.0
2021-01-03,35.0,35.0,11.0,9.0,16.0,2.0,59.0,0.0,45.0,7.0,0.0,171.0
2021-01-04,30.0,24.0,5.0,6.0,7.0,2.0,17.0,0.0,17.0,7.0,0.0,128.0
2021-01-05,29.0,26.0,6.0,3.0,7.0,6.0,12.0,0.0,6.0,7.0,0.0,99.0
2021-01-07,28.0,21.0,3.0,3.0,7.0,3.0,15.0,0.0,15.0,6.0,0.0,109.0


article,BAGUETTE,BANETTE,BOULE 400G,CAMPAGNE,CEREAL BAGUETTE,COOKIE,CROISSANT,ECLAIR,PAIN AU CHOCOLAT,SPECIAL BREAD,TARTELETTE,TRADITIONAL BAGUETTE
date,,,,,,,,,,,,
2021-01-02,41.4,42.00,16.5,18.0,23.75,8.0,72.6,0.0,57.6,19.2,0.0,153.6
2021-01-03,31.5,36.75,16.5,16.2,20.00,2.0,64.9,0.0,54.0,16.8,0.0,205.2
2021-01-04,27.0,25.20,7.5,10.8,8.75,2.0,18.7,0.0,20.4,16.8,0.0,153.6
2021-01-05,26.1,27.30,9.0,5.4,8.75,6.0,13.2,0.0,7.2,16.8,0.0,118.8
2021-01-07,25.2,22.05,4.5,5.4,8.75,3.0,16.5,0.0,18.0,14.4,0.0,130.8
